7.1 Start with a baseline

Before building any ML model, create naive baselines to beat: (1) season average attendance, (2) same fixture last year, (3) average by opponent category. If your model cannot beat these, you need to revisit your features.

In [6]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

match = pd.read_csv("data/gold_match.csv")
tickets = pd.read_csv("data/gold_match_tickets.csv")
context = pd.read_csv("data/gold_match_context.csv")

match = match[match["is_home_match"] == True]

df = match.merge(tickets, on="match_id", how="left")
df = df.merge(context, on="match_id", how="left")

if "match_date_x" in df.columns:
    df = df.rename(columns={"match_date_x": "match_date"})
elif "match_date_y" in df.columns:
    df = df.rename(columns={"match_date_y": "match_date"})

df = df.sort_values(["away_team", "match_date"])

df["season_avg"] = df.groupby("season")["tickets_scanned"].transform("mean")
mae_season = mean_absolute_error(df["tickets_scanned"], df["season_avg"])

df["fixture_last_year"] = df.groupby("away_team")["tickets_scanned"].shift(1)
df_fixture = df.dropna(subset=["fixture_last_year"])
mae_fixture = mean_absolute_error(
    df_fixture["tickets_scanned"],
    df_fixture["fixture_last_year"]
)

df["opponent_avg"] = df.groupby("away_team")["tickets_scanned"].transform("mean")
mae_opponent = mean_absolute_error(df["tickets_scanned"], df["opponent_avg"])

print("Baseline 1 - Season Average MAE:", mae_season)
print("Baseline 2 - Same Fixture MAE:", mae_fixture)
print("Baseline 3 - Opponent Average MAE:", mae_opponent)

Baseline 1 - Season Average MAE: 1398.8086637471893
Baseline 2 - Same Fixture MAE: 2220.64
Baseline 3 - Opponent Average MAE: 1249.7046948356808
